# Xarray-spatial
### User Guide: Contour Line Extraction
-----

Contour lines (isolines) connect points of equal value on a surface. They are one of the most common ways to represent elevation on topographic maps.

The `contours` function uses a marching squares algorithm to trace isolines through a 2D raster at specified elevation values. It supports NumPy, CuPy, Dask, and Dask+CuPy backends.

**Topics covered:**
- [Basic contour extraction](#Basic-Contour-Extraction)
- [Automatic level selection](#Automatic-Level-Selection)
- [GeoDataFrame output](#GeoDataFrame-Output)
- [Overlaying contours on terrain](#Contours-on-Terrain)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import xrspatial
from xrspatial import contours
from xrspatial.terrain import generate_terrain

## Generate Synthetic Terrain

We start by generating a synthetic elevation raster using `generate_terrain`.

In [ ]:
W, H = 400, 300
terrain = xr.DataArray(np.zeros((H, W)))
terrain = generate_terrain(terrain)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(terrain.values, cmap='terrain', origin='lower')
plt.colorbar(im, ax=ax, label='Elevation')
ax.set_title('Synthetic Terrain')
plt.tight_layout()
plt.show()

## Basic Contour Extraction

Extract contour lines at specific elevation levels. The function returns a list of `(level, coordinates)` tuples, where each `coordinates` array has shape `(N, 2)` with `(row, col)` positions.

In [ ]:
# Pick a few specific levels
vmin, vmax = float(np.nanmin(terrain.values)), float(np.nanmax(terrain.values))
levels = np.linspace(vmin, vmax, 12)[1:-1]  # 10 interior levels

lines = contours(terrain, levels=levels)
print(f'Extracted {len(lines)} contour polylines across {len(levels)} levels')

# Show the first result
level, coords = lines[0]
print(f'First line: level={level:.1f}, {len(coords)} vertices')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(terrain.values, cmap='terrain', origin='lower', alpha=0.5)

# Color contours by level
unique_levels = sorted(set(lvl for lvl, _ in lines))
cmap = plt.cm.inferno
norm = plt.Normalize(vmin=min(unique_levels), vmax=max(unique_levels))

for level, coords in lines:
    ax.plot(coords[:, 1], coords[:, 0], color=cmap(norm(level)), linewidth=0.8)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
plt.colorbar(sm, ax=ax, label='Elevation')
ax.set_title('Contour Lines on Terrain')
plt.tight_layout()
plt.show()

## Automatic Level Selection

When `levels` is not specified, the function automatically picks `n_levels` evenly spaced values between the raster's min and max.

In [ ]:
# Let the function choose 20 levels automatically
auto_lines = contours(terrain, n_levels=20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(terrain.values, cmap='terrain', origin='lower', alpha=0.4)

for level, coords in auto_lines:
    ax.plot(coords[:, 1], coords[:, 0], 'k-', linewidth=0.5)

ax.set_title('Automatic Contour Levels (n_levels=20)')
plt.tight_layout()
plt.show()

## GeoDataFrame Output

Set `return_type='geopandas'` to get a GeoDataFrame with `level` and `geometry` columns. This is useful for further spatial analysis or export to GIS formats.

In [ ]:
gdf = contours(terrain, levels=levels, return_type='geopandas')
print(f'GeoDataFrame with {len(gdf)} contour lines')
gdf.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
gdf.plot(ax=ax, column='level', cmap='coolwarm', linewidth=0.8, legend=True)
ax.set_title('Contour GeoDataFrame')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Contours on Terrain

Combine hillshade with contour lines for a topographic map look.

In [ ]:
hillshade = terrain.xrs.hillshade()

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(hillshade.values, cmap='gray', origin='lower')
ax.imshow(terrain.values, cmap='terrain', origin='lower', alpha=0.35)

# Draw contours with labeled index contours (every 5th level thicker)
for i, (level, coords) in enumerate(auto_lines):
    lw = 1.2 if i % 5 == 0 else 0.4
    ax.plot(coords[:, 1], coords[:, 0], 'sienna', linewidth=lw, alpha=0.7)

ax.set_title('Topographic Map: Hillshade + Contours')
ax.set_xlabel('Column')
ax.set_ylabel('Row')
plt.tight_layout()
plt.show()

## Using the Accessor

The `contours` function is also available via the `.xrs` accessor on any DataArray.

In [ ]:
# Equivalent to contours(terrain, levels=[500])
accessor_lines = terrain.xrs.contours(levels=levels[:3])
print(f'{len(accessor_lines)} contour lines via .xrs accessor')

### References
- Marching squares algorithm: https://en.wikipedia.org/wiki/Marching_squares
- Contour lines in cartography: https://en.wikipedia.org/wiki/Contour_line